# FPViT a 224 px su DermaMNIST-C — Colab

Addestra FPViT a **224×224** sul dataset **DermaMNIST-C** (Abhishek et al., split per lesione, senza leakage)
e lo testa anche su **DermaMNIST-E** (test ISIC 2018, lesioni diverse: test esterno).

Due configurazioni, **3 seed ciascuna**, come ablation:

| nome | estrattore ResNet-18 | ricetta |
| --- | --- | --- |
| `scratch` | inizializzazione casuale (come nel paper) | AdamW lr 1e-3, warmup 5, fino a 100 epoche |
| `pretrained` | pesi ImageNet di torchvision (niente dati dermatologici) | AdamW lr 3e-4, backbone ×0.1, warmup 3, fino a 60 epoche |

Comuni a entrambe: mixed precision, gradient clipping 1.0, class weighting `effective`, selezione del checkpoint su
**balanced accuracy di validation**, early stopping, augmentation `default` (gruppo diedrale + crop leggero + colore tenue).

**Regola del test**: il test (celle 9) si esegue **una volta sola, alla fine**, dopo aver scelto tutto sulla validation.
Guardare il test e poi cambiare configurazione lo trasforma in una seconda validation.

Prepara lo zip dal Mac (circa 60 MB: senza `.venv/`, `output/` e dataset; resta incluso il `best_model.pt` di `model/baseline/baseline_paper`)
e caricalo in `MyDrive/pyramid_project.zip`:

```bash
cd /path/to/DermaMNIST-agentic_project-main
rm -f ~/Desktop/pyramid_project.zip
zip -r ~/Desktop/pyramid_project.zip . -x '.venv/*' '.git/*' 'output/*' 'input/uploads/*' \
    'database/dermamnist/*' 'model/promoted/*' '*__pycache__*' '*.DS_Store' '*/last_model.pt'
```

In alternativa, se il codice è su un repository git con questa stessa struttura, metti `CODE = "git"` e il suo `GIT_URL` nella cella 1.

## 1 — GPU, Drive, codice

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime senza GPU: Runtime > Change runtime type > GPU"
print(torch.__version__, "|", torch.cuda.get_device_name(0))

from google.colab import drive; drive.mount('/content/drive')
DRIVE      = "/content/drive/MyDrive/agenticderma"
DRIVE_DATA = f"{DRIVE}/data"            # npz scaricati una volta sola (C-224 1.1 GB, E-224 1.3 GB)
DATA       = "/content/data"            # copia locale: leggere 1 GB da Drive a ogni run e' lento
RUNS       = "/content/runs_224"        # disco locale; sincronizzato su Drive dalla cella 5
DRIVE_RUNS = f"{DRIVE}/runs_224"
!mkdir -p {DRIVE_DATA} {DATA} {RUNS} {DRIVE_RUNS}
# Sessione nuova: /content e' vuota. Recupera i run gia' fatti PRIMA di qualsiasi sync.
!rsync -a {DRIVE_RUNS}/ {RUNS}/

CODE = "zip"                            # "zip" (MyDrive/pyramid_project.zip) oppure "git"
GIT_URL = ""                            # repository con la struttura di questa cartella (system/, model/, ...)
%cd /content
!rm -rf pyramid_project
if CODE == "zip":
    !mkdir pyramid_project && unzip -q "/content/drive/MyDrive/pyramid_project.zip" -d pyramid_project
else:
    !git clone -q {GIT_URL} pyramid_project
%cd /content/pyramid_project
!grep -q "dermamnist_c" system/nets/dataset.py && echo "codice con supporto DermaMNIST-C: ok"
import sys; sys.path.insert(0, "/content/pyramid_project/system")

## 2 — Dipendenze

Colab ha già torch, torchvision, numpy, scikit-learn e tqdm. Manca solo `medmnist`.
**Non eseguire `pip install -r requirements.txt`**: pip può sostituire il torch CUDA con una wheel CPU-only.

In [ ]:
!pip install -q medmnist
import torch
assert torch.cuda.is_available(), "CUDA sparita: pip ha sostituito torch. Riavvia il runtime."

## 3 — Dati: DermaMNIST-C e DermaMNIST-E da Zenodo

Il download avviene una volta sola su Drive, con verifica MD5, poi i file si copiano sul disco locale di Colab.
I test di C ed E si usano **solo** alla cella 9. Le versioni a 28 px servono solo a rivalutare `baseline_paper` sugli stessi test senza leakage.

In [ ]:
from nets.dataset import dataset_file
for ds, size in [("dermamnist_c", 224), ("dermamnist_e", 224), ("dermamnist_c", 28), ("dermamnist_e", 28)]:
    path = dataset_file(ds, size, data_root=DRIVE_DATA)      # scarica se manca, MD5 verificato
    print(ds, size, "->", path)
!rsync -a --info=progress2 {DRIVE_DATA}/ {DATA}/
!ls -lh {DATA}

In [ ]:
import numpy as np
with np.load(f"{DATA}/dermamnist_corrected_224.npz") as z:
    sizes = {s: len(z[f"{s}_labels"]) for s in ("train", "val")}
    counts = np.bincount(z["train_labels"].reshape(-1), minlength=7).tolist()
    sample = z["val_images"][:8]
print("split:", sizes, "(test non letto qui)")
print("train per classe:", counts)
assert sizes == {"train": 8215, "val": 573}

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 8, figsize=(16, 2.4))
for a, im in zip(ax, sample): a.imshow(im); a.axis("off")
plt.suptitle("DermaMNIST-C val, 224 px: devono essere nitide, non pixelate"); plt.show()

## 4 — Quanto costa un'epoca su questa GPU

Due epoche della configurazione `pretrained`, in una cartella usa e getta. Se il tempo per epoca è dominato dai dati
(la GPU resta sotto il 50% in `nvidia-smi`), il collo di bottiglia sono le 2 CPU di Colab, non il modello.

In [ ]:
import json, pathlib, time
!python system/train.py --dataset dermamnist_c --img-size 224 --pretrained \
    --optimizer adamw --lr 3e-4 --backbone-lr-mult 0.1 --warmup-epochs 1 --epochs 2 \
    --amp --grad-clip 1.0 --batch-size 64 --num-workers 2 \
    --data-root {DATA} --out /content/probe --index-file none --no-progress
rec = json.loads(pathlib.Path("/content/probe/experiment_record.json").read_text())
sec = sum(h["epoch_seconds"] for h in rec["history"]) / len(rec["history"])
print(f"\n{sec:.0f} s/epoca -> {sec*60/3600:.1f} h per 60 epoche, {sec*100/3600:.1f} h per 100")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 5 — Sincronizzazione su Drive (lanciala prima del training)

`last_model.pt` viene riscritto a ogni epoca: tenerlo direttamente su Drive rallenterebbe tutto. Per questo i run
restano sul disco locale e un processo in background li copia su Drive ogni 15 minuti.
Se il runtime viene riciclato perdi al massimo 15 minuti. Il sync **non** usa `--delete`.

In [ ]:
!nohup bash -c 'while true; do rsync -a {RUNS}/ {DRIVE_RUNS}/; sleep 900; done' > /content/sync.log 2>&1 &
print("sync attivo:", RUNS, "->", DRIVE_RUNS, "ogni 15 min")

## 6 — Gli esperimenti

La cella si può **rieseguire identica dopo una disconnessione**. Per ogni run:
- se è già finito (`completed` o `early_stop`) lo salta;
- se esiste `last_model.pt` riprende con `--resume`;
- altrimenti parte da zero.

`MAX_SECONDS` > 0 chiude il run in modo pulito prima del limite della sessione, poi lo si riprende rieseguendo la cella.

In [ ]:
import json, pathlib

COMMON = ("--dataset dermamnist_c --img-size 224 --batch-size 64 --amp --grad-clip 1.0 "
          "--class-weight effective --select-on balanced_acc --aug-preset default "
          "--weight-decay 0.05 --optimizer adamw --num-workers 2 --no-progress --log-file "
          f"--data-root {DATA} --index-file {RUNS}/index.jsonl")
CONFIGS = {
    "scratch":    "--lr 1e-3 --warmup-epochs 5 --epochs 100 --early-stop-patience 20",
    "pretrained": "--pretrained --lr 3e-4 --backbone-lr-mult 0.1 --warmup-epochs 3 "
                  "--epochs 60 --early-stop-patience 15",
}
SEEDS = [42, 43, 44]
ORDER = ["pretrained", "scratch"]       # prima quella che si prevede migliore
MAX_SECONDS = 0                          # es. 3*3600 per chiudere pulito prima del limite Colab

def run_dir(name, seed): return pathlib.Path(RUNS) / f"fpvit_c224_{name}_s{seed}"

def status(d):
    rec = d / "experiment_record.json"
    if rec.exists() and json.loads(rec.read_text())["stop_reason"] in ("completed", "early_stop"):
        return "done"
    return "resume" if (d / "last_model.pt").exists() else "new"

for name in ORDER:
    for seed in SEEDS:
        d = run_dir(name, seed); st = status(d)
        print(f"\n===== {d.name}: {st} =====")
        if st == "done":
            continue
        cmd = (f"python system/train.py {COMMON} {CONFIGS[name]} --seed {seed} --out {d} "
               f"--max-seconds {MAX_SECONDS}" + (" --resume" if st == "resume" else ""))
        get_ipython().system(cmd)
!rsync -a {RUNS}/ {DRIVE_RUNS}/

## 7 — Le curve (validation)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for name, color in (("scratch", "tab:blue"), ("pretrained", "tab:orange")):
    for seed in SEEDS:
        f = run_dir(name, seed) / "metrics.csv"
        if not f.exists(): continue
        df = pd.read_csv(f); lbl = f"{name} s{seed}"
        ax[0].plot(df.epoch, df.val_balanced_acc, color=color, alpha=.8, label=lbl)
        ax[1].plot(df.epoch, df.train_loss, color=color, alpha=.8, ls="--")
        ax[1].plot(df.epoch, df.val_loss, color=color, alpha=.8)
        ax[2].plot(df.epoch, df.lr, color=color, alpha=.8)
ax[0].set_title("val balanced accuracy"); ax[0].legend(fontsize=8)
ax[1].set_title("loss (tratteggio = train, pesata; continua = val)"); ax[2].set_title("lr (teste)")
for a in ax: a.set_xlabel("epoca")
plt.show()

## 8 — Riepilogo sulla validation (media ± std sui seed)

Questo è il confronto da usare per **decidere**. La recall per classe mostra se le classi rare
(dermatofibroma, lesioni vascolari) vengono imparate davvero.

In [ ]:
import numpy as np
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
rows = []
for name in ORDER:
    for seed in SEEDS:
        f = run_dir(name, seed) / "experiment_record.json"
        if not f.exists(): continue
        r = json.loads(f.read_text())
        best = next(h for h in r["history"] if h["epoch"] == r["best_epoch"])
        rows.append({"config": name, "seed": seed, "stop": r["stop_reason"], "best_epoch": r["best_epoch"],
                     "val_bal_acc": best["val_balanced_acc"], "val_macro_auc": best["val_macro_auc"],
                     "val_macro_f1": best["val_macro_f1"], "val_acc": best["val_acc"],
                     **{f"rec_{c}": v for c, v in zip(CLASSES, best["val_per_class_recall"])}})
val = pd.DataFrame(rows)
display(val.round(3))
metrics = ["val_bal_acc", "val_macro_auc", "val_macro_f1", "val_acc"] + [f"rec_{c}" for c in CLASSES]
summary = val.groupby("config")[metrics].agg(["mean", "std"]).round(3)
display(summary)

## 9 — Test finale (una volta sola)

Per ogni checkpoint, due test:
- **DermaMNIST-C test**: 1227 immagini, stessa distribuzione del training, lesioni mai viste;
- **DermaMNIST-E test**: 1511 immagini, test ISIC 2018, **esterno**.

Come riferimento, `baseline_paper` (FPViT a 28 px sullo split ufficiale) viene valutato sugli stessi due test a 28 px.
È un confronto pulito: le lesioni del test di C sono state tolte dal train ufficiale, e il test di E (ISIC 2018)
non ha lesioni in comune con HAM10000.

In [ ]:
CONFIRM_FINAL_TEST = False              # mettere True solo quando la scelta sulla validation e' chiusa
assert CONFIRM_FINAL_TEST, "Test finale non confermato: vedi la regola del test in cima al notebook."

tests = []
def run_test(ckpt, dataset, label, seed):
    out = pathlib.Path(ckpt).with_name(f"test_report_{dataset}.json")
    get_ipython().system(f'python system/evaluate.py --checkpoint "{ckpt}" --dataset {dataset} '
                         f'--data-root {DATA} --num-workers 2 --out "{out}" > /dev/null')
    r = json.loads(out.read_text())
    tests.append({"model": label, "seed": seed, "test": dataset, "bal_acc": r["test_balanced_acc"],
                  "macro_auc": r["test_macro_auc"], "macro_f1": r["test_macro_f1"], "acc": r["test_acc"],
                  **{f"rec_{c}": v for c, v in zip(CLASSES, r["per_class_recall"])}})

for name in ORDER:
    for seed in SEEDS:
        ckpt = run_dir(name, seed) / "best_model.pt"
        if ckpt.exists():
            for ds in ("dermamnist_c", "dermamnist_e"):
                run_test(ckpt, ds, f"fpvit224_{name}", seed)
for ds in ("dermamnist_c", "dermamnist_e"):
    run_test("model/baseline/baseline_paper/best_model.pt", ds, "baseline_paper_28", 42)

test = pd.DataFrame(tests)
display(test.groupby(["test", "model"])[["bal_acc", "macro_auc", "macro_f1", "acc"] +
                                        [f"rec_{c}" for c in CLASSES]].agg(["mean", "std"]).round(3))
test.to_csv(f"{RUNS}/final_test_summary.csv", index=False)
!rsync -a {RUNS}/ {DRIVE_RUNS}/

## 10 — Portare i modelli sul Mac

Scarica da Drive la cartella `agenticderma/runs_224/` e mettila nel progetto come `output/runs_224/`.
Il `ModelZoo` **non** legge `output/runs_224/`, di proposito: un modello entra nell'ensemble solo passando dal gate di
promozione (validation comune DermaMNIST-C val e decisione umana):

```bash
python run.py promote output/runs_224/fpvit_c224_pretrained_s42
```

Nel frattempo si possono usare singolarmente:

```bash
python run.py predict --image input/samples_224 --labels input/samples_224/labels.csv \
    --checkpoint output/runs_224/fpvit_c224_pretrained_s42/best_model.pt
```

(le immagini di `input/samples/` sono a 28 px: per un modello a 224 servono immagini a 224, come quelle di `input/samples_224/`.)